In [1]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.8/32.8 MB 48.2 MB/s eta 0:00:00


In [1]:
#This code will work irrespective of the double and triple bonds
# Import all the dependencies
from rdkit import Chem,DataStructs
from rdkit.Chem import AllChem
import pandas as pd
# Wite the input file containing reactions in form of SMILE and divide it into four columns:
#Reactant 1,Reactant 2,Product 1,Product 2
df=pd.read_csv(r"ARE Ouput (ERI Input).csv")
#Loop over each reaction
for index in df.index:
      reactant1_smiles=df["Reactant 1"][index].strip()
      reactant2_smiles=df["Reactant 2"][index].strip()
      product1_smiles=df["Product 1"][index].strip()
      product2_smiles=df["Product 2"][index].strip()
      if reactant1_smiles == '*':                               #Replace * with a blank string
        reactant1_smiles = reactant1_smiles.replace('*','')
      if reactant2_smiles == '*':
        reactant2_smiles=reactant2_smiles.replace('*','')
      if product1_smiles == '*':
        product1_smiles = product1_smiles.replace('*','')
      if product2_smiles == '*':
        product2_smiles=product2_smiles.replace('*','')
      # Convert SMILES strings to RDKit molecules
      reactant1_mol = Chem.MolFromSmiles(reactant1_smiles)
      reactant2_mol = Chem.MolFromSmiles(reactant2_smiles)
      product1_mol = Chem.MolFromSmiles(product1_smiles)
      product2_mol = Chem.MolFromSmiles(product2_smiles)
      #calculate the number of Carbon and Oxygen atoms in each reactants and products
      num_C_R1= sum(1 for atom in reactant1_mol.GetAtoms() if atom.GetSymbol()=="C")
      num_C_R2= sum(1 for atom in reactant2_mol.GetAtoms() if atom.GetSymbol()=="C")
      num_C_P1= sum(1 for atom in product1_mol.GetAtoms() if atom.GetSymbol()=="C")
      num_C_P2= sum(1 for atom in product2_mol.GetAtoms() if atom.GetSymbol()=="C")
      num_O_R1= sum(1 for atom in reactant1_mol.GetAtoms() if atom.GetSymbol()=="O")
      num_O_R2= sum(1 for atom in reactant2_mol.GetAtoms() if atom.GetSymbol()=="O")
      num_O_P1= sum(1 for atom in product1_mol.GetAtoms() if atom.GetSymbol()=="O")
      num_O_P2= sum(1 for atom in product2_mol.GetAtoms() if atom.GetSymbol()=="O")
      #Generate the reaction smiles strings
      if len(reactant1_smiles) > 0 and len(reactant2_smiles) > 0:
          reactant_smiles = reactant1_smiles+'.'+reactant2_smiles
      elif (len(reactant1_smiles)>0):
          reactant_smiles = reactant1_smiles
      else:
          reactant_smiles = reactant2_smiles

      if (len(product1_smiles)>0 and len(product2_smiles)>0):
          product_smiles = product1_smiles+'.'+product2_smiles
      elif (len(product1_smiles)>0):
          product_smiles = product1_smiles
      else:
          product_smiles = product2_smiles
      # Convert the reaction SMILES strings into Mol
      reactant_mol = Chem.MolFromSmiles(reactant_smiles)
      product_mol = Chem.MolFromSmiles(product_smiles)
      # Introduce the Morgan Fingerprints and Tanimoto Similarity
      fp_r1 = AllChem.GetMorganFingerprintAsBitVect(reactant1_mol, 2)
      fp_r2 = AllChem.GetMorganFingerprintAsBitVect(reactant2_mol, 2)
      fp_p1 = AllChem.GetMorganFingerprintAsBitVect(product1_mol, 2)
      fp_p2 = AllChem.GetMorganFingerprintAsBitVect(product2_mol, 2)
      # define similarity with respect to  the number of carbon atoms
      similarity11= abs(num_C_R1-num_C_P1)
      similarity12= abs(num_C_R1-num_C_P2)
      similarity21= abs(num_C_R2-num_C_P1)
      similarity22= abs(num_C_R2-num_C_P2)
      # define tanimoto similarity in case the similarities are equal and coupling and decoupling reactions
      if (similarity11==similarity12==similarity21==similarity22) and (reactant1_smiles !='' and reactant2_smiles !='' and product1_smiles !='' and product2_smiles !=''):
        similarity11 = DataStructs.TanimotoSimilarity(fp_r1, fp_p1)
        similarity12 = DataStructs.TanimotoSimilarity(fp_r1, fp_p2)
        similarity21 = DataStructs.TanimotoSimilarity(fp_r2, fp_p1)
        similarity22 = DataStructs.TanimotoSimilarity(fp_r2, fp_p2)
        max_similarity=max(similarity11,similarity21,similarity12,similarity22) # find the max similarity in case of tanimoto based similarity
      else:
        max_similarity=min(similarity11,similarity21,similarity12,similarity22) # find the min similarity in case of carbon-number based similarity
      # Add Hydrogen to Each Reactant and Product Mol
      reactant_mol = Chem.AddHs(reactant_mol)
      reactant1_mol = Chem.AddHs(reactant1_mol)
      reactant2_mol = Chem.AddHs(reactant2_mol)
      product_mol = Chem.AddHs(product_mol)
      product1_mol = Chem.AddHs(product1_mol)
      product2_mol = Chem.AddHs(product2_mol)
      # Logic 1
      # Define reactant and product bond dictionary
      reactant_bond_dict = {}
      for bond in reactant_mol.GetBonds():
          atom1 = bond.GetBeginAtom().GetSymbol()
          atom2 = bond.GetEndAtom().GetSymbol()

          bond_key = frozenset([atom1, atom2])
          if bond_key in reactant_bond_dict:
              reactant_bond_dict[bond_key] += 1
          else:
              reactant_bond_dict[bond_key] = 1

      reactant1_bond_dict = {}
      for bond in reactant1_mol.GetBonds():
          atom1 = bond.GetBeginAtom().GetSymbol()
          atom2 = bond.GetEndAtom().GetSymbol()

          bond_key = frozenset([atom1, atom2])
          if bond_key in reactant1_bond_dict:
              reactant1_bond_dict[bond_key] += 1
          else:
              reactant1_bond_dict[bond_key] = 1

      reactant2_bond_dict = {}
      for bond in reactant2_mol.GetBonds():
          atom1 = bond.GetBeginAtom().GetSymbol()
          atom2 = bond.GetEndAtom().GetSymbol()

          bond_key = frozenset([atom1, atom2])
          if bond_key in reactant2_bond_dict:
              reactant2_bond_dict[bond_key] += 1
          else:
              reactant2_bond_dict[bond_key] = 1

      product_bond_dict = {}
      for bond in product_mol.GetBonds():
          atom1 = bond.GetBeginAtom().GetSymbol()
          atom2 = bond.GetEndAtom().GetSymbol()

          bond_key = frozenset([atom1, atom2])
          if bond_key in product_bond_dict:
              product_bond_dict[bond_key] += 1
          else:
              product_bond_dict[bond_key] = 1

      product1_bond_dict = {}
      for bond in product1_mol.GetBonds():
          atom1 = bond.GetBeginAtom().GetSymbol()
          atom2 = bond.GetEndAtom().GetSymbol()

          bond_key = frozenset([atom1, atom2])
          if bond_key in product1_bond_dict:
              product1_bond_dict[bond_key] += 1
          else:
              product1_bond_dict[bond_key] = 1

      product2_bond_dict = {}
      for bond in product2_mol.GetBonds():
          atom1 = bond.GetBeginAtom().GetSymbol()
          atom2 = bond.GetEndAtom().GetSymbol()

          bond_key = frozenset([atom1, atom2])
          if bond_key in product2_bond_dict:
              product2_bond_dict[bond_key] += 1
          else:
              product2_bond_dict[bond_key] = 1
      # Count the number of bonds broken and formed and see that they are both <= 1
      bonds_broken = 0
      for bond in reactant_bond_dict:
          if (bond not in product_bond_dict.keys()):
              product_bond_dict[bond]=0
          if (reactant_bond_dict[bond]>=product_bond_dict[bond]):
              bonds_broken += reactant_bond_dict[bond]-product_bond_dict[bond]

      bonds_formed = 0
      for bond in product_bond_dict:
          if (bond not in reactant_bond_dict.keys()):
              reactant_bond_dict[bond]=0
          if (product_bond_dict[bond]>=reactant_bond_dict[bond]):
              bonds_formed += product_bond_dict[bond]-reactant_bond_dict[bond]
      # Initiate the reaction as elementary if the no. of bonds broken and formed is <=1
      if (bonds_broken<=1 and bonds_formed<=1):
          still_valid = 1
      # Classify the reaction as non-elementary otherwise
      else:
          still_valid = 0

      #Logic 2
      # Define all the carbon-oxygen ordered pair vectors
      oxygen_switched_carbon=0
      carbon_oxygen_neighbors_reactant1 = []
      for atom in reactant1_mol.GetAtoms():
          if atom.GetSymbol() == 'C':
              symbol = atom.GetSymbol()
              neighboring_oxygens = [neighbor.GetIdx() for neighbor in atom.GetNeighbors() if neighbor.GetSymbol() == 'O']
              num_oxygen_atoms = len(neighboring_oxygens)
              carbon_oxygen_neighbors_reactant1.append((symbol, num_oxygen_atoms))
      carbon_oxygen_neighbors_reactant2 = []
      for atom in reactant2_mol.GetAtoms():
          if atom.GetSymbol() == 'C':
              symbol = atom.GetSymbol()
              neighboring_oxygens = [neighbor.GetIdx() for neighbor in atom.GetNeighbors() if neighbor.GetSymbol() == 'O']
              num_oxygen_atoms = len(neighboring_oxygens)
              carbon_oxygen_neighbors_reactant2.append((symbol, num_oxygen_atoms))
      carbon_oxygen_neighbors_product1 = []
      for atom in product1_mol.GetAtoms():
          if atom.GetSymbol() == 'C':
              symbol = atom.GetSymbol()
              neighboring_oxygens = [neighbor.GetIdx() for neighbor in atom.GetNeighbors() if neighbor.GetSymbol() == 'O']
              num_oxygen_atoms = len(neighboring_oxygens)
              carbon_oxygen_neighbors_product1.append((symbol, num_oxygen_atoms))
      carbon_oxygen_neighbors_product2 = []
      for atom in product2_mol.GetAtoms():
          if atom.GetSymbol() == 'C':
              symbol = atom.GetSymbol()
              neighboring_oxygens = [neighbor.GetIdx() for neighbor in atom.GetNeighbors() if neighbor.GetSymbol() == 'O']
              num_oxygen_atoms = len(neighboring_oxygens)
              carbon_oxygen_neighbors_product2.append((symbol, num_oxygen_atoms))
      # Define the Reactants and Products List
      carbon_oxygen_neighbors_reactant12_list=carbon_oxygen_neighbors_reactant1+carbon_oxygen_neighbors_reactant2
      carbon_oxygen_neighbors_reactant21_list=carbon_oxygen_neighbors_reactant2+carbon_oxygen_neighbors_reactant1
      carbon_oxygen_neighbors_product12_list=carbon_oxygen_neighbors_product1+carbon_oxygen_neighbors_product2
      carbon_oxygen_neighbors_product21_list=carbon_oxygen_neighbors_product2+carbon_oxygen_neighbors_product1
      # Based on the maximum similaity, find out the number of oxygen bond broken or formed
      # isomerization reactions
      if (reactant1_smiles=='' or reactant2_smiles=='') and (product1_smiles=='' or product2_smiles==''):
        if carbon_oxygen_neighbors_reactant12_list==carbon_oxygen_neighbors_product21_list:
          oxygen_switched_carbon=0
        else:
          oxygen_switched_list=[]
          for i in range(len(carbon_oxygen_neighbors_reactant12_list)):
            no_of_oxygen_switched=carbon_oxygen_neighbors_product21_list[i][1]-carbon_oxygen_neighbors_reactant12_list[i][1]
            oxygen_switched_list.append(no_of_oxygen_switched)
          for i in range(len(oxygen_switched_list)):
            oxygen_switched_list[i]=abs(oxygen_switched_list[i])
          no_of_oxygen_switched=1
          if oxygen_switched_list.count(1)>2:
            still_valid=0
          if oxygen_switched_list.count(1) == 2:
            for i in range(len(oxygen_switched_list)-2):
              if oxygen_switched_list[i]==1:
                if oxygen_switched_list[i+1]==1:
                  no_of_oxygen_switched==1
                if oxygen_switched_list[i+2]==1:
                  no_of_oxygen_switched==1
                else:
                  still_valid=0
          for i in range(len(oxygen_switched_list)):
            if abs(oxygen_switched_list[i])>1:
              still_valid=0
          if no_of_oxygen_switched==1:
            oxygen_switched_carbon=1
      else:
        oxygen_switched_carbon1=2
        oxygen_switched_carbon2=2
        oxygen_switched_carbon3=2
        oxygen_switched_carbon4=2
        oxygen_switched_carbonA=1
        oxygen_switched_carbonB=1
        oxygen_switched_carbonC=1
        oxygen_switched_carbonD=1
        no_O_switched_places1=3
        no_O_switched_places2=3
        no_O_switched_places3=3
        no_O_switched_places4=3
        if max_similarity==similarity21:
          if carbon_oxygen_neighbors_reactant21_list==carbon_oxygen_neighbors_product12_list:
            oxygen_switched_carbon1=0
          else:
            no_O_switched_places1 = 0
            for i in range(len(carbon_oxygen_neighbors_reactant21_list)):
              if carbon_oxygen_neighbors_reactant21_list[i] !=carbon_oxygen_neighbors_product12_list[i]:
                no_O_switched_places1 += abs(carbon_oxygen_neighbors_product12_list[i][1] - carbon_oxygen_neighbors_reactant21_list[i][1])
            if no_O_switched_places1<=2:
              oxygen_switched_carbon1=1
            if no_O_switched_places1<=1:
              oxygen_switched_carbonA=0.5
        if max_similarity==similarity12:
            if carbon_oxygen_neighbors_reactant12_list==carbon_oxygen_neighbors_product21_list:
              oxygen_switched_carbon2=0
            else:
              no_O_switched_places2 = 0
              for i in range(len(carbon_oxygen_neighbors_reactant12_list)):
                if carbon_oxygen_neighbors_reactant12_list[i] !=carbon_oxygen_neighbors_product21_list[i]:
                  no_O_switched_places2 += abs(carbon_oxygen_neighbors_product21_list[i][1] - carbon_oxygen_neighbors_reactant12_list[i][1])
              if no_O_switched_places2<=2:
                oxygen_switched_carbon2 = 1
              if no_O_switched_places2<=1:
                oxygen_switched_carbonB=0.5
        if max_similarity==similarity11:
            if carbon_oxygen_neighbors_reactant12_list==carbon_oxygen_neighbors_product12_list:
              oxygen_switched_carbon3=0
            else:
              no_O_switched_places3 = 0
              for i in range(len(carbon_oxygen_neighbors_reactant12_list)):
                if carbon_oxygen_neighbors_reactant12_list[i] !=carbon_oxygen_neighbors_product12_list[i]:
                  no_O_switched_places3 += abs(carbon_oxygen_neighbors_product12_list[i][1] - carbon_oxygen_neighbors_reactant12_list[i][1])
              if no_O_switched_places3<=2:
                oxygen_switched_carbon3=1
              if no_O_switched_places3<=1:
                oxygen_switched_carbonC=0.5
        if max_similarity==similarity22:
            if carbon_oxygen_neighbors_reactant21_list==carbon_oxygen_neighbors_product21_list:
              oxygen_switched_carbon4=0
            else:
              no_O_switched_places4 = 0
              for i in range(len(carbon_oxygen_neighbors_reactant21_list)):
                if carbon_oxygen_neighbors_reactant21_list[i] !=carbon_oxygen_neighbors_product21_list[i]:
                  no_O_switched_places4 += abs(carbon_oxygen_neighbors_product21_list[i][1] - carbon_oxygen_neighbors_reactant21_list[i][1])
              if no_O_switched_places4<=2:
                oxygen_switched_carbon4=1
              if no_O_switched_places4<=1:
                oxygen_switched_carbonD=0.5
        if oxygen_switched_carbon4==0 or oxygen_switched_carbon3==0 or oxygen_switched_carbon2==0 or oxygen_switched_carbon1==0:
          oxygen_switched_carbon=0
        else:
          if oxygen_switched_carbon4==1 or oxygen_switched_carbon3==1 or oxygen_switched_carbon2==1 or oxygen_switched_carbon1==1:
            oxygen_switched_carbon=1
          if no_O_switched_places4>2 and no_O_switched_places3>2 and no_O_switched_places2>2 and no_O_switched_places1>2:
            still_valid=0
        if oxygen_switched_carbon==1 and (oxygen_switched_carbonA==0.5 or oxygen_switched_carbonB==0.5 or oxygen_switched_carbonC==0.5 or oxygen_switched_carbonD==0.5):
            oxygen_switched_carbon=0.5
        if oxygen_switched_carbon==0:
          if oxygen_switched_carbon4==0:
            similarity22=5
          if oxygen_switched_carbon3==0:
            similarity11=5
          if oxygen_switched_carbon2==0:
            similarity12 = 5
          if  oxygen_switched_carbon1==0:
            similarity21 = 5
        if oxygen_switched_carbon==1 or oxygen_switched_carbon==0.5:
          if oxygen_switched_carbon4==1:
            similarity22=5
          if oxygen_switched_carbon3==1:
            similarity11=5
          if oxygen_switched_carbon2==1:
            similarity12 = 5
          if  oxygen_switched_carbon1==1:
            similarity21 = 5
      max_similarity= max(similarity22,similarity21,similarity11,similarity12)
      #Logic 3
      # Define all the ordered pair vectors
      hydrogen_switched_molecule=0
      reactant1_list = []
      for atom in reactant1_mol.GetAtoms():
            if atom.GetSymbol() != 'H':
                symbol = atom.GetSymbol()
                neighboring_atoms = [neighbor.GetIdx() for neighbor in atom.GetNeighbors() if neighbor.GetSymbol() == 'H']
                num_hydrogen_atoms = len(neighboring_atoms)
                reactant1_list.append((symbol, num_hydrogen_atoms))
      reactant2_list = []
      for atom in reactant2_mol.GetAtoms():
            if atom.GetSymbol() != 'H':
                symbol = atom.GetSymbol()
                neighboring_atoms = [neighbor.GetIdx() for neighbor in atom.GetNeighbors() if neighbor.GetSymbol() == 'H']
                num_hydrogen_atoms = len(neighboring_atoms)
                reactant2_list.append((symbol, num_hydrogen_atoms))
      product1_list = []
      for atom in product1_mol.GetAtoms():
            if atom.GetSymbol() != 'H':
                symbol = atom.GetSymbol()
                neighboring_atoms = [neighbor.GetIdx() for neighbor in atom.GetNeighbors() if neighbor.GetSymbol() == 'H']
                num_hydrogen_atoms = len(neighboring_atoms)
                product1_list.append((symbol, num_hydrogen_atoms))
      product2_list = []
      for atom in product2_mol.GetAtoms():
            if atom.GetSymbol() != 'H':
                symbol = atom.GetSymbol()
                neighboring_atoms = [neighbor.GetIdx() for neighbor in atom.GetNeighbors() if neighbor.GetSymbol() == 'H']
                num_hydrogen_atoms = len(neighboring_atoms)
                product2_list.append((symbol, num_hydrogen_atoms))
      # Define all the Reactants and Products List
      reactant_list=reactant1_list+reactant2_list
      product_list=product1_list+product2_list
      reactant21_list=reactant2_list+reactant1_list
      product12_list=product1_list+product2_list
      reactant12_list=reactant1_list+reactant2_list
      product21_list=product2_list+product1_list
      reactant21_list=reactant2_list+reactant1_list
      product12_list=product1_list+product2_list
      reactant12_list=reactant1_list+reactant2_list
      product21_list=product2_list+product1_list
      total_H_reactant=sum(i[1] for i in reactant_list)
      total_H_product=sum(i[1] for i in product_list)
      reactant_oxygen_list=[]
      reactant_carbon_list=[]
      for element in reactant_list:
          if element[0] == 'C':
              reactant_carbon_list.append(element)
          elif element[0] == 'O':
              reactant_oxygen_list.append(element)
      product_oxygen_list=[]
      product_carbon_list=[]
      for element in product_list:
          if element[0] == 'C':
              product_carbon_list.append(element)
          elif element[0] == 'O':
              product_oxygen_list.append(element)
      reactant12_oxygen_list=[]
      reactant12_carbon_list=[]
      for element in reactant12_list:
          if element[0] == 'C':
              reactant12_carbon_list.append(element)
          elif element[0] == 'O':
              reactant12_oxygen_list.append(element)
      product12_oxygen_list=[]
      product12_carbon_list=[]
      for element in product12_list:
          if element[0] == 'C':
              product12_carbon_list.append(element)
          elif element[0] == 'O':
              product12_oxygen_list.append(element)
      reactant21_oxygen_list=[]
      reactant21_carbon_list=[]
      for element in reactant21_list:
          if element[0] == 'C':
              reactant21_carbon_list.append(element)
          elif element[0] == 'O':
              reactant21_oxygen_list.append(element)
      product21_oxygen_list=[]
      product21_carbon_list=[]
      for element in product21_list:
          if element[0] == 'C':
              product21_carbon_list.append(element)
          elif element[0] == 'O':
              product21_oxygen_list.append(element)
      #Based on the maximum similaity, find out the number of hydrogen bond broken or formed
      # if total_H_reactant==total_H_product==0: # If there is no hydrogen in reactant and products, hydrogen won't switch places
      #   hydrogen_switched_molecule=0
      # Isomerization Reactions
      if (reactant1_smiles=='' or reactant2_smiles=='') and (product1_smiles=='' or product2_smiles==''):
        if product_list==reactant_list:
          hydrogen_switched_molecule=0
        else:
          hydrogen_switched_listO=[]
          for i in range(len(reactant_oxygen_list)):
            no_of_hydrogen_switched=product_oxygen_list[i][1]-reactant_oxygen_list[i][1]
            hydrogen_switched_listO.append(no_of_hydrogen_switched)
          hydrogen_switched_listC=[]
          for i in range(len(reactant_carbon_list)):
            no_of_hydrogen_switchedC=product_carbon_list[i][1]-reactant_carbon_list[i][1]
            hydrogen_switched_listC.append(no_of_hydrogen_switchedC)
          hydrogen_switched_list=hydrogen_switched_listC+hydrogen_switched_listO
          for i in range(len(hydrogen_switched_list)):
            hydrogen_switched_list[i]=abs(hydrogen_switched_list[i])
          for i in range(len(hydrogen_switched_listC)):
            hydrogen_switched_listC[i]=abs(hydrogen_switched_listC[i])
          for i in range(len(hydrogen_switched_listO)):
            hydrogen_switched_listO[i]=abs(hydrogen_switched_listO[i])
          no_of_carbon_switched=1
          if hydrogen_switched_list.count(1)>2:
            still_valid=0
          elif (hydrogen_switched_listC.count(1) == 2):
            indices_of_one = [i for i, value in enumerate(hydrogen_switched_listC) if value ==1]
            carbon_place_1 = indices_of_one[0]+1
            carbon_place_2 = indices_of_one[1]+1
            carbon_count_1 = 0
            for atom in reactant1_mol.GetAtoms():
                if atom.GetSymbol() == 'C':
                    carbon_count_1 += 1
                    if carbon_count_1 == carbon_place_1:
                        carbon_index_1 = atom.GetIdx()
                        break
            carbon_count_2 = 0
            for atom in reactant1_mol.GetAtoms():
                if atom.GetSymbol() == 'C':
                    carbon_count_2 += 1
                    if carbon_count_2 == carbon_place_2:
                        carbon_index_2 = atom.GetIdx()
                        break
            if reactant2_smiles=="":
              dist_matrix = AllChem.GetDistanceMatrix(reactant1_mol)
              number_of_bonds = dist_matrix[carbon_index_1][carbon_index_2]
            if reactant1_smiles=="":
              dist_matrix = AllChem.GetDistanceMatrix(reactant2_mol)
              number_of_bonds = dist_matrix[carbon_index_1][carbon_index_2]
            if number_of_bonds > 2:
              still_valid=0
          elif (hydrogen_switched_listO.count(1) == 2):
            indices_of_one = [i for i, value in enumerate(hydrogen_switched_listO) if value ==1]
            oxygen_place_1 = indices_of_one[0]+1
            oxygen_place_2 = indices_of_one[1]+1
            oxygen_count_1 = 0
            for atom in reactant1_mol.GetAtoms():
                if atom.GetSymbol() == 'O':
                    oxygen_count_1 += 1
                    if oxygen_count_1 == oxygen_place_1:
                        oxygen_index_1 = atom.GetIdx()
                        break
            oxygen_count_2 = 0
            for atom in reactant1_mol.GetAtoms():
                if atom.GetSymbol() == 'O':
                    oxygen_count_2 += 1
                    if oxygen_count_2 == oxygen_place_2:
                        oxygen_index_2 = atom.GetIdx()
                        break
            if reactant2_smiles=="":
              dist_matrix = AllChem.GetDistanceMatrix(reactant1_mol)
              number_of_bonds = dist_matrix[oxygen_index_1][oxygen_index_2]
            if reactant1_smiles=="":
              dist_matrix = AllChem.GetDistanceMatrix(reactant2_mol)
              number_of_bonds = dist_matrix[oxygen_index_1][oxygen_index_2]
            if number_of_bonds > 2:
              still_valid=0
          elif (hydrogen_switched_list.count(1) == 2):
            indices_of_one_C = [i for i, value in enumerate(hydrogen_switched_listC) if value ==1]
            indices_of_one_O = [i for i, value in enumerate(hydrogen_switched_listO) if value ==1]
            carbon_place= indices_of_one_C[0]+1
            oxygen_place= indices_of_one_O[0]+1
            carbon_count=0
            for atom in reactant1_mol.GetAtoms():
                if atom.GetSymbol() == 'C':
                    carbon_count += 1
                    if carbon_count == carbon_place:
                        carbon_index = atom.GetIdx()
                        break
            oxygen_count=0
            for atom in reactant1_mol.GetAtoms():
                if atom.GetSymbol() == 'O':
                    oxygen_count += 1
                    if oxygen_count == oxygen_place:
                        oxygen_index = atom.GetIdx()
                        break
            if reactant2_smiles=="":
              dist_matrix = AllChem.GetDistanceMatrix(reactant1_mol)
              number_of_bonds = dist_matrix[oxygen_index][carbon_index]
            if reactant1_smiles=="":
              dist_matrix = AllChem.GetDistanceMatrix(reactant2_mol)
              number_of_bonds = dist_matrix[oxygen_index][carbon_index]
            if number_of_bonds > 2:
              still_valid=0
          for i in range(len(hydrogen_switched_list)):
            if abs(hydrogen_switched_list[i])>1:
              still_valid=0
          if no_of_carbon_switched==1:
            hydrogen_switched_molecule=1
          if no_of_carbon_switched==0:
            still_valid=0

      else:
        hydrogen_switched_molecule4=1
        hydrogen_switched_molecule3=1
        hydrogen_switched_molecule2=1
        hydrogen_switched_molecule1=1
        hydrogen_switched_moleculeA=1
        hydrogen_switched_moleculeB=1
        hydrogen_switched_moleculeC=1
        hydrogen_switched_moleculeD=1
        no_h_switched_places4=3
        no_h_switched_places3=3
        no_h_switched_places2=3
        no_h_switched_places1=3
        if max_similarity==similarity21:
          if (reactant21_oxygen_list==product12_oxygen_list) and (reactant21_carbon_list==product12_carbon_list):
                hydrogen_switched_molecule1=0
          else:
            no_h_switched_places_O = 0
            for i in range(len(reactant21_oxygen_list)):
              no_h_switched_places_O += abs(product12_oxygen_list[i][1] - reactant21_oxygen_list[i][1])

            no_h_switched_places_C = 0
            for i in range(len(reactant21_carbon_list)):
              no_h_switched_places_C += abs(product12_carbon_list[i][1] - reactant21_carbon_list[i][1])
            no_h_switched_places1=no_h_switched_places_C+no_h_switched_places_O
            if no_h_switched_places1<=2:
              hydrogen_switched_molecule1=1
            if no_h_switched_places1<=1:
              hydrogen_switched_moleculeA=0.5
        if max_similarity==similarity11:
              if reactant12_oxygen_list==product12_oxygen_list and reactant12_carbon_list==product12_carbon_list:
                  hydrogen_switched_molecule2=0
              else:
                no_h_switched_places_O = 0
                for i in range(len(reactant12_oxygen_list)):
                  no_h_switched_places_O += abs(product12_oxygen_list[i][1] - reactant12_oxygen_list[i][1])

                no_h_switched_places_C = 0
                for i in range(len(reactant12_carbon_list)):
                  no_h_switched_places_C += abs(product12_carbon_list[i][1] - reactant12_carbon_list[i][1])
                no_h_switched_places2=no_h_switched_places_C+no_h_switched_places_O
                if no_h_switched_places2<=2:
                  hydrogen_switched_molecule2=1
                if no_h_switched_places2<=1:
                  hydrogen_switched_moleculeB=0.5

        if max_similarity==similarity22:
            if (reactant21_oxygen_list==product21_oxygen_list) and (reactant21_carbon_list==product21_carbon_list):
              hydrogen_switched_molecule3=0
            else:
              no_h_switched_places_O = 0
              for i in range(len(reactant21_oxygen_list)):
                no_h_switched_places_O += abs(product21_oxygen_list[i][1] - reactant21_oxygen_list[i][1])

              no_h_switched_places_C = 0
              for i in range(len(reactant21_carbon_list)):
                no_h_switched_places_C += abs(product21_carbon_list[i][1] - reactant21_carbon_list[i][1])
              no_h_switched_places3=no_h_switched_places_C+no_h_switched_places_O
              if no_h_switched_places3<=2:
                hydrogen_switched_molecule3=1
              if no_h_switched_places3<=1:
                hydrogen_switched_moleculeC=0.5
        if max_similarity==similarity12:
              if reactant12_oxygen_list==product21_oxygen_list and reactant12_carbon_list==product21_carbon_list:
                hydrogen_switched_molecule4=0
              else:
                  no_h_switched_places_O = 0
                  for i in range(len(reactant12_oxygen_list)):
                    no_h_switched_places_O += abs(product21_oxygen_list[i][1] - reactant12_oxygen_list[i][1])

                  no_h_switched_places_C = 0
                  for i in range(len(reactant12_carbon_list)):
                    no_h_switched_places_C += abs(product21_carbon_list[i][1] - reactant12_carbon_list[i][1])
                  no_h_switched_places4=no_h_switched_places_C+no_h_switched_places_O
                  if no_h_switched_places4<=2:
                    hydrogen_switched_molecule4=1
                  if no_h_switched_places4<=1:
                    hydrogen_switched_moleculeD=0.5

        if hydrogen_switched_molecule4==0 or hydrogen_switched_molecule3==0 or hydrogen_switched_molecule2==0 or hydrogen_switched_molecule1==0:
          hydrogen_switched_molecule=0
        else:
          if hydrogen_switched_molecule4==1 and hydrogen_switched_molecule3==1 and hydrogen_switched_molecule2==1 and hydrogen_switched_molecule1==1:
            hydrogen_switched_molecule=1
          if no_h_switched_places4>2 and no_h_switched_places3>2 and no_h_switched_places2>2 and no_h_switched_places1>2:
            still_valid=0
        if hydrogen_switched_molecule==1 and (hydrogen_switched_moleculeA==0.5 or hydrogen_switched_moleculeB==0.5 or hydrogen_switched_moleculeC==0.5 or hydrogen_switched_moleculeD==0.5):
            hydrogen_switched_molecule=0.5
      #logic 4 breakage and formation of C-C bonds
      Carbon_bonds_break_or_formed=0
      if (num_C_R1==0) and (num_C_R2>0):
        if num_C_P1>0 and num_C_P2>0:
          Carbon_bonds_break_or_formed=1
      elif (num_C_R2==0) and (num_C_R1>0):
        if num_C_P1>0 and num_C_P2>0:
          Carbon_bonds_break_or_formed=1
      elif (num_C_P1==0) and (num_C_P2>0):
        if num_C_R1>0 and num_C_R2>0:
          Carbon_bonds_break_or_formed=1
      elif (num_C_P2==0) and (num_C_P1>0):
        if num_C_R1>0 and num_C_R2>0:
          Carbon_bonds_break_or_formed=1
      else:
        A=num_C_R1-num_C_P1
        B=num_C_R1-num_C_P2
        C=num_C_R2-num_C_P2
        D=num_C_R2-num_C_P1

        if (A!=0 and B!=0 and C!=0 and D != 0):
          Carbon_bonds_break_or_formed=1
      # Both logic 2,logic 3 and logic 4 cannot occur together
      if (oxygen_switched_carbon==1 and hydrogen_switched_molecule==1) or (oxygen_switched_carbon==1 and Carbon_bonds_break_or_formed==1) or (hydrogen_switched_molecule==1 and Carbon_bonds_break_or_formed==1) or (hydrogen_switched_molecule==0.5 and oxygen_switched_carbon==1) or (hydrogen_switched_molecule==1 and oxygen_switched_carbon==0.5) or ((oxygen_switched_carbon==0.5 and Carbon_bonds_break_or_formed==1 and hydrogen_switched_molecule==0.5)) or (hydrogen_switched_molecule==0.5 and oxygen_switched_carbon==0.5) or (Carbon_bonds_break_or_formed==1 and hydrogen_switched_molecule==0.5) or (Carbon_bonds_break_or_formed==1 and oxygen_switched_carbon==0.5):
          still_valid=0
      df.loc[index, "Is the reaction elementary"] =bool(still_valid)

C:\Users\USER\AppData\Local\Temp\ipykernel_21588\3582650881.py:634: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'True' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[index, "Is the reaction elementary"] =bool(still_valid)
[15:04:14] WARNING: not removing hydrogen atom without neighbors
[15:04:14] WARNING: not removing hydrogen atom without neighbors
[15:04:15] WARNING: not removing hydrogen atom without neighbors
[15:04:15] WARNING: not removing hydrogen atom without neighbors
[15:04:15] WARNING: not removing hydrogen atom without neighbors
[15:04:15] WARNING: not removing hydrogen atom without neighbors
[15:04:15] WARNING: not removing hydrogen atom without neighbors
[15:04:15] WARNING: not removing hydrogen atom without neighbors
[15:04:15] WARNING: not removing hydrogen atom without neighbors
[15:04:15] WARNING: not removing hydrogen atom without neighbors
[15:

In [2]:
df.value_counts('Is the reaction elementary')

Is the reaction elementary
False    95334
True      9389
Name: count, dtype: int64